In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data/raw/phmsa"

for f in sorted(RAW.rglob("*")):
    if f.suffix.lower() in {".xlsx", ".xls", ".txt", ".csv"}:
        print("\n===", f.name, "===")
        if f.suffix.lower() in {".xlsx", ".xls"}:
            sheets = pd.read_excel(f, sheet_name=None, nrows=5)
            for name, df in sheets.items():
                print(f"sheet '{name}': {df.shape[1]} columns")
                print("  cause/narrative columns:",
                      [c for c in df.columns if any(k in str(c).upper() for k in ("CAUSE", "NARRATIVE"))])
        else:
            df = pd.read_csv(f, sep=None, engine="python", nrows=5, encoding="latin-1")
            print(f"{df.shape[1]} columns")
            print("  cause/narrative columns:",
                  [c for c in df.columns if any(k in str(c).upper() for k in ("CAUSE", "NARRATIVE"))])


=== accident_hazardous_liquid_jan2010_present.txt ===
648 columns
  cause/narrative columns: ['CAUSE', 'CAUSE_DETAILS', 'ROOT_CAUSE_CATEGORY', 'ROOT_CAUSE_TYPE', 'ROOT_CAUSE_TYPE_OTHER', 'NARRATIVE']


In [2]:
path = RAW / "accident_hazardous_liquid_jan2010_present.txt"

# Detect the separator from the header line (tab, pipe or comma)
first_line = path.open(encoding="latin-1").readline()
sep = max(["\t", "|", ","], key=first_line.count)
print("Separator:", repr(sep))

df = pd.read_csv(path, sep=sep, encoding="latin-1", low_memory=False)
print("Shape:", df.shape)
print("Date/year columns:", [c for c in df.columns if "DATE" in c.upper() or "YEAR" in c.upper()][:10])

print("\n--- CAUSE ---")
print(df["CAUSE"].value_counts(dropna=False).to_string())

print("\n--- CAUSE_DETAILS (top 15) ---")
print(df["CAUSE_DETAILS"].value_counts().head(15).to_string())

print("\n--- ROOT_CAUSE columns filled in ---")
print(df[["ROOT_CAUSE_CATEGORY", "ROOT_CAUSE_TYPE"]].notna().mean().round(2).to_string())

Separator: '\t'
Shape: (6052, 648)
Date/year columns: ['REPORT_RECEIVED_DATE', 'IYEAR', 'LOCAL_DATETIME', 'INCIDENT_IDENTIFIED_DATETIME', 'SHUTDOWN_DATETIME', 'RESTART_DATETIME', 'INITIAL_RESPONDER_COM_DATETIME', 'ON_SITE_DATETIME', 'CONFIRMED_DISCOVERY_DATETIME', 'NRC_RPT_DATETIME']

--- CAUSE ---
CAUSE
EQUIPMENT FAILURE                   2759
CORROSION FAILURE                   1328
INCORRECT OPERATION                  834
MATERIAL FAILURE OF PIPE OR WELD     406
NATURAL FORCE DAMAGE                 264
EXCAVATION DAMAGE                    199
OTHER ACCIDENT CAUSE                 134
OTHER OUTSIDE FORCE DAMAGE           128

--- CAUSE_DETAILS (top 15) ---
CAUSE_DETAILS
INTERNAL CORROSION                                                                                                                  853
NON-THREADED CONNECTION FAILURE                                                                                                     735
PUMP OR PUMP-RELATED EQUIPMENT                  

In [3]:
narr = df["NARRATIVE"].fillna("").str.strip()
print("Share of reports with no narrative:", round((narr == "").mean(), 3))
print("\nNarrative length in words:")
print(narr[narr != ""].str.split().str.len().describe().round(0).to_string())

for text in df["NARRATIVE"].dropna().sample(2, random_state=0):
    print("\n---\n", text[:700])

Share of reports with no narrative: 0.005

Narrative length in words:
count    6021.0
mean      154.0
std       108.0
min         1.0
25%        81.0
50%       125.0
75%       194.0
max       688.0

---
 AT 10:40 AM CST ON JULY 17, 2012, THE BUCKEYE CONTROL CENTER DETECTED A SIGNIFICANT PRESSURE DROP ON THE GT254EE PIPELINE, A 10" LINE RUNNING FROM GRANVILLE STATION TO ELKHART LAKE PUMP STATION. THE CONTROLLER IMMEDIATELY RESPONDED AND SHUT DOWN THE LINE. FOOT AND AIR PATROLS WERE DISPATCHED, AND LOCAL AND FEDERAL AGENCIES WERE NOTIFIED OF A POTENTIAL RELEASE. THE SITE WAS NOT LOCATED ON JULY 17. FOOT AND AIR PATROLS RESUMED ON JULY 18, AND THE RELEASE WAS LOCATED BY A FOOT PATROL TEAM AT 11:15 CST IN JACKSON, WI, WASHINGTON COUNTY, APPROXIMATELY 10 MILES NORTH OF GRANVILLE STATION. THERE WAS NO PRODUCT AT THE SURFACE, BUT HIGH VAPOR CONTENT PRESENT AT THE SCENE. ALL LOCAL AND FEDERAL AGENCI

---
 THE WILBREEZE PUMP STATION UNIT# 2 WAS SHUTDOWN AND  ISOLATED  TO CONDUCT MAINTENANCE ON 

In [4]:
nar = df["NARRATIVE"].fillna("")
chars = nar.str.len()
print("Longest narratives (characters):", chars.nlargest(5).tolist())
print("Reports at the maximum length:", int((chars == chars.max()).sum()))
print("Reports within 20 characters of the maximum:", int((chars >= chars.max() - 20).sum()))

Longest narratives (characters): [4000, 4000, 4000, 4000, 4000]
Reports at the maximum length: 5
Reports within 20 characters of the maximum: 19


In [5]:
print(df["IYEAR"].value_counts().sort_index().to_string())

IYEAR
2010    350
2011    344
2012    367
2013    400
2014    455
2015    460
2016    420
2017    415
2018    405
2019    384
2020    332
2021    347
2022    295
2023    299
2024    287
2025    269
2026    223


In [6]:
giveaways = {
    "CORROSION FAILURE": "CORRO",
    "EXCAVATION DAMAGE": "EXCAVAT",
    "NATURAL FORCE DAMAGE": "LIGHTNING|FLOOD|EARTHQUAKE|FREEZ|RAIN|HURRICANE",
    "INCORRECT OPERATION": "PROCEDURE|OPERATOR ERROR|HUMAN ERROR|INCORRECT",
}
for cause, pattern in giveaways.items():
    hit = nar.str.contains(pattern, regex=True)
    in_class = hit[df["CAUSE"] == cause].mean()
    in_others = hit[df["CAUSE"] != cause].mean()
    print(f"{cause:22s} keyword in own class: {in_class:.0%}   in other classes: {in_others:.0%}")

CORROSION FAILURE      keyword in own class: 66%   in other classes: 3%
EXCAVATION DAMAGE      keyword in own class: 61%   in other classes: 18%
NATURAL FORCE DAMAGE   keyword in own class: 70%   in other classes: 22%
INCORRECT OPERATION    keyword in own class: 38%   in other classes: 7%


## Arm A baseline: TF-IDF (word 1–2 grams + char 3–5 grams) + logistic regression

Time split: train 2010–2021 (4,650), validation 2022–2023 (593), test 2024–2026 (778). C chosen on validation; test scored once.

| Model | Test macro-F1 | Test accuracy |
|---|---|---|
| Dummy (most frequent) | 0.078 | 0.454 |
| TF-IDF + LR | 0.609 | 0.794 |
| TF-IDF + LR, giveaways masked | 0.590 | 0.790 |

- Strong baseline; accuracy vs macro-F1 gap means some small classes are weak.
- Masking giveaway words costs only 0.019 macro-F1: causes are described with a rich vocabulary beyond the masked words. The shortcut test is too narrow to prove understanding.
- Small test classes (19–36 documents) make per-class F1 noisy; "Other accident cause" grows from ~2% of training data to ~5% of test (distribution shift revealed by the time split).

In [8]:
R = ROOT / "reports"
print(pd.read_csv(R / "phmsa_tfidf_original_report.csv", index_col=0).round(2).to_string())

pred = pd.read_parquet(R / "phmsa_tfidf_original_predictions.parquet")
cm = pd.crosstab(pred["label"], pred["pred"], normalize="index").round(2)
cm


                                  precision  recall  f1-score  support
CORROSION FAILURE                      0.85    0.91      0.88   204.00
EQUIPMENT FAILURE                      0.84    0.87      0.86   353.00
EXCAVATION DAMAGE                      0.71    0.89      0.79    19.00
INCORRECT OPERATION                    0.64    0.72      0.68    69.00
MATERIAL FAILURE OF PIPE OR WELD       0.66    0.73      0.69    48.00
NATURAL FORCE DAMAGE                   0.57    0.57      0.57    28.00
OTHER ACCIDENT CAUSE                   0.50    0.06      0.10    36.00
OTHER OUTSIDE FORCE DAMAGE             0.67    0.19      0.30    21.00
accuracy                               0.79    0.79      0.79     0.79
macro avg                              0.68    0.62      0.61   778.00
weighted avg                           0.78    0.79      0.77   778.00


pred,CORROSION FAILURE,EQUIPMENT FAILURE,EXCAVATION DAMAGE,INCORRECT OPERATION,MATERIAL FAILURE OF PIPE OR WELD,NATURAL FORCE DAMAGE,OTHER ACCIDENT CAUSE,OTHER OUTSIDE FORCE DAMAGE
label,,,,,,,,
CORROSION FAILURE,0.91,0.05,0.00,0.00,0.03,0.00,0.00,0.00
EQUIPMENT FAILURE,0.02,0.87,0.00,0.06,0.03,0.01,0.01,0.00
EXCAVATION DAMAGE,0.11,0.00,0.89,0.00,0.00,0.00,0.00,0.00
INCORRECT OPERATION,0.00,0.25,0.00,0.72,0.00,0.03,0.00,0.00
MATERIAL FAILURE OF PIPE OR WELD,0.10,0.15,0.00,0.00,0.73,0.02,0.00,0.00
NATURAL FORCE DAMAGE,0.00,0.36,0.00,0.04,0.00,0.57,0.00,0.04
OTHER ACCIDENT CAUSE,0.50,0.22,0.03,0.08,0.06,0.03,0.06,0.03
OTHER OUTSIDE FORCE DAMAGE,0.05,0.33,0.29,0.05,0.00,0.10,0.00,0.19


### Per-class results and confusions (TF-IDF, original)

| Cause | Precision | Recall | F1 | Test docs |
|---|---|---|---|---|
| Corrosion | 0.85 | 0.91 | 0.88 | 204 |
| Equipment failure | 0.84 | 0.87 | 0.86 | 353 |
| Excavation | 0.71 | 0.89 | 0.79 | 19 |
| Incorrect operation | 0.64 | 0.72 | 0.68 | 69 |
| Material failure | 0.66 | 0.73 | 0.69 | 48 |
| Natural force | 0.57 | 0.57 | 0.57 | 28 |
| Other accident cause | 0.50 | 0.06 | 0.10 | 36 |
| Other outside force | 0.67 | 0.19 | 0.30 | 21 |

- Physically sensible confusions: incorrect operation → equipment (25%; human action vs hardware), natural force → equipment (36%; trigger vs damaged component), material failure → equipment/corrosion (15%/10%), other outside force → excavation (29%; both external impacts). These need cause-and-effect reading, not vocabulary.
- Unexpected: "Other accident cause" → corrosion 50% (recall 6%). Hypothesis: "other" often means the cause was undetermined at filing time, while the narrative discusses a suspected cause. Checking sub-causes and narratives.

In [9]:
gold = pd.read_parquet(ROOT / "data/gold/phmsa_cause.parquet")
test = gold[gold["split"] == "test"].reset_index(drop=True)
other = test[test["label"] == "OTHER ACCIDENT CAUSE"]

print("Sub-causes of 'Other accident cause' in the test set:")
print(other["detail"].value_counts().to_string())

mistaken = other[pred.loc[other.index, "pred"].values == "CORROSION FAILURE"]
for text in mistaken["text"].head(3):
    print("\n---\n", text[:900])

Sub-causes of 'Other accident cause' in the test set:
detail
UNKNOWN          33
MISCELLANEOUS     3

---
 ON 9/18/2026 AT 10:26, THE LOCAL FIRE DEPARTMENT CONTACTED THE CONTROL CENTER REPORTING OIL SURFACING FROM THE GROUND NEAR THE 13002 CMAR-CTOL-16INCH LINE. LINE WAS IMMEDIATELY SHUTDOWN AND ISOLATED. THE LINE WAS EXCAVATED AND THE RELEASE CONFIRMED TO BE FROM THE 13002 CMAR-CTOL-16INCH LINE. APPROXIMATELY 24 BARRELS OF CRUDE OIL WAS RELEASED. AN OSRO WAS MOBILIZED TO CLEANUP FREE PRODUCT AND CONTAMINATED SOIL. ON 9/21/26, THE FAILED PIPE WAS CUTOUT AND REPLACED. THE FAILED PIPE WAS SENT TO A THIRD PARTY LAB FOR ANALYSIS. THE CAUSE OF THE LEAK IS STILL UNDER INVESTIGATION. A SUPPLEMENTAL REPORT WILL BE SUBMITTED ONCE THE INVESTIGATION IS COMPLETE.

---
 ON 9/15/2026, AT APPROXIMATELY 17:31, AN OPERATOR EMPLOYEE DISCOVERED A CRUDE OIL LEAK ON TANK 2721 16-INCH FILL LINE VALVE AT MIDLAND NORTH TANK FARM. THE TANK WAS ISOLATED AND PRODUCT DRAINED. THE CRUDE OIL WAS FOUND COMING FROM A

### "Other accident cause" in the test set = cause not yet determined

- 33 of 36 test cases have sub-cause UNKNOWN; narratives state the cause is "still under investigation" with a supplemental report to follow.
- TF-IDF predicts corrosion for many of them: the narratives describe pinhole leaks and pipes sent to a lab, language typical of older corrosion reports.
- Likely mechanism (reporting lag): initial reports are filed within 30 days, often before the investigation ends; supplemental reports later update the cause. Older incidents have been updated, recent ones have not, so "unknown" is over-represented in the test years.

**Decision:** evaluate on two tracks:
1. Determined causes (main): the 7 physical cause categories, excluding UNKNOWN sub-causes.
2. Pending detection (side task): does the narrative say the cause is not yet known?

In [10]:
pending = gold["text"].str.contains(
    r"UNDER INVESTIGATION|INVESTIGATION IS ONGOING|INVESTIGATION IS STILL|SUPPLEMENTAL REPORT WILL", regex=True)
gold["pending_text"] = pending
gold["unknown_label"] = gold["detail"] == "UNKNOWN"

print("Share of narratives with 'pending' language, by split:")
print(gold.groupby("split")["pending_text"].mean().round(3).to_string())

print("\nPending language vs UNKNOWN label (counts):")
print(pd.crosstab(gold["pending_text"], gold["unknown_label"]).to_string())

print("\nConfirmed causes among narratives with pending language:")
print(gold[gold["pending_text"] & ~gold["unknown_label"]]["label"].value_counts().to_string())

Share of narratives with 'pending' language, by split:
split
test     0.165
train    0.038
val      0.074

Pending language vs UNKNOWN label (counts):
unknown_label  False  True 
pending_text               
False           5644     30
True             327     20

Confirmed causes among narratives with pending language:
label
CORROSION FAILURE                   143
EQUIPMENT FAILURE                   106
MATERIAL FAILURE OF PIPE OR WELD     34
INCORRECT OPERATION                  22
OTHER OUTSIDE FORCE DAMAGE            6
EXCAVATION DAMAGE                     6
NATURAL FORCE DAMAGE                  5
OTHER ACCIDENT CAUSE                  5


### Pending-investigation language vs labels

- Share of narratives with pending language: train 3.8%, validation 7.4%, test 16.5%, consistent with reporting lag.
- 347 narratives contain pending language; only 20 have an UNKNOWN label, while 327 carry a confirmed cause (corrosion 143, equipment 106, material 34, …): the label was updated by a supplemental report. Whether the narrative was also updated is being checked.
- 30 UNKNOWN cases have no pending language under our regex, partly due to phrasing variants ("an investigation into the cause of the release is ongoing"): fixed patterns are brittle.

In [11]:
noisy = gold[gold["pending_text"] & ~gold["unknown_label"]]
for _, r in noisy.sample(4, random_state=1).iterrows():
    print(f"\n--- label: {r['label']} | split: {r['split']} ---\n...{r['text'][-700:]}")


--- label: EQUIPMENT FAILURE | split: test ---
...PIPE WITH THE REINFORCEMENT SLEEVE WILL BE SENT TO A THIRD PARTY LABORATORY FOR ANALYSIS. A SUPPLEMENTAL REPORT WILL BE SUBMITTED ONCE THE INVESTIGATION IS COMPLETE. SUPPLEMENTAL & FINAL: THE CAUSE OF THE ACCIDENT WAS DUE TO A THREADED TOR CONNECTION FAILURE AT A PREVIOUSLY INSTALLED STOPPLE FITTING. THE RESULT OF THE CAUSE INVESTIGATION WERE INCONCLUSIVE. THE LAB WAS UNABLE TO FIND A LEAK PATH AT THE WELDS OF THE TORS OR ENCIRCLEMENT SLEEVES, AND NO THROUGH WALL FEATURE OR OTHER LEAK PATH WAS LOCATED. THE STOPPLE AND TOR FITTINGS AT THIS SITE WERE INSTALLED UNDER THE PRIOR OWNERSHIP OF THIS LINE. TO PREVENT RECURRENCE, ALL STOPPLES AND THE ASSOCIATED TOR FITTINGS WERE REMOVED FROM THE LINE.

--- label: EQUIPMENT FAILURE | split: train ---
...ING PREPARATION FOR TANK VENT VALVE INSPECTION. AN ADJUSTABLE WRENCH WAS IN USE AND STRUCK A HARD OBJECT WHILE VAPORS WERE ESCAPING FROM THE ACCESS PANEL PERIMETER. THIS RESULTED IN AN IGNITION OF

### Supplemental narratives: updated, not noisy

- Sampled narratives with pending language but a confirmed cause all contain an appended conclusion ("SUPPLEMENTAL & FINAL: the cause was…", "SUPPLEMENTAL NARRATIVE: the leak was caused by…"). The cause is in the text, at the end. These 327 examples stay.
- Pending language does not mean pending: ground truth for pending detection is the UNKNOWN sub-cause, not keywords.
- The conclusion is often the last sentence: if the SLM must shorten narratives, keep the end.

**Tracks**
- Main: 7 physical causes ("Other accident cause" removed: in test, 33 UNKNOWN + 3 miscellaneous).
- Side: pending detection (sub-cause UNKNOWN) vs everything else; keyword rule and TF-IDF baselines.

### Baselines on both tracks

| Task | Model | Score |
|---|---|---|
| Main (7 causes) | Dummy | macro-F1 0.092, accuracy 0.476 |
| Main | TF-IDF + LR | macro-F1 0.730, accuracy 0.829 |
| Main | TF-IDF + LR, masked | macro-F1 0.692, accuracy 0.832 |
| Pending | Keyword rule | precision 0.148, recall 0.606, F1 0.238 |
| Pending | TF-IDF + LR | F1 0.000 (never predicts pending) |

- Removing undeterminable cases raised macro-F1 by 12 points: part of the earlier weakness was narratives without a cause.
- Keyword rule cannot distinguish "pending" from "was pending, then concluded" (supplemental conclusions appended after the old sentence).
- TF-IDF has only 17 pending examples to learn from (0 in validation): a reading-comprehension task with almost no training data, where a zero-shot language model should have an advantage.

## Arm B: Qwen3-4B-Instruct-2507, zero-shot (Colab T4, 4-bit)

Prompt: narrative plus one-line definitions of the 7 causes (from PHMSA's form categories); answer with a single letter. Validation checked first; test not yet used.

| Model | Validation macro-F1 | Validation accuracy |
|---|---|---|
| TF-IDF + LR | 0.716 | – |
| Qwen3-4B zero-shot | 0.594 | 0.698 |

- 0 invalid answers.
- Zero-shot is ~12 points below TF-IDF: the SLM understands narratives but does not know PHMSA's labeling conventions, which TF-IDF learned from 4,650 labeled reports. Fine-tuning (QLoRA) is designed to close that gap.

### Zero-shot errors on validation (prompt v1)

- Over-attributes human error: 22% of equipment failures labeled incorrect operation (incorrect operation precision 0.48). Narratives mention workers near failing equipment.
- Falls for repair excavations: excavation recall 1.00 but precision 0.34; 20% of material failures and 8% of corrosion labeled excavation.
- Misfiles weather: natural force recall 0.37 (26% → other outside force).
- Strong on corrosion (F1 0.87); equipment precision 0.95.
- Errors reflect PHMSA labeling conventions rather than comprehension.

**Prompt v2:** conventions written into the definitions (human error only when it caused the release; repair excavations are not excavation damage; weather is natural force). One revision only, evaluated on validation; the better prompt is then run once on test.

### Prompt v2 (conventions written in): worse on validation

| Prompt | Validation macro-F1 | Accuracy |
|---|---|---|
| v1 | 0.594 | 0.698 |
| v2 | 0.550 | 0.639 |

- Natural force improved (F1 0.52 → 0.63), but the excavation warning overcorrected (recall 1.00 → 0.43) and broadening "other outside force" to damage "by people" pulled equipment failures into it (precision 0.15).
- Small models are sensitive to wording; fixing one confusion moved errors elsewhere. One revision only, to avoid fitting the prompt to validation.
- Decision: prompt v1 used for the single test run. Fine-tuning is the principled way to teach conventions.

### Test results: TF-IDF vs zero-shot Qwen3-4B (prompt v1)

**Main track (7 causes)**

| Model | Macro-F1 | Accuracy |
|---|---|---|
| TF-IDF + LR | 0.730 | 0.829 |
| Qwen3-4B zero-shot | 0.632 | 0.705 |

Zero-shot macro recall 0.75 vs precision 0.62: without knowledge of class frequencies, it over-predicts rare causes (excavation recall 0.95, precision 0.46) and under-predicts equipment failure (recall 0.59).

**Pending detection**

| Model | Precision | Recall | F1 |
|---|---|---|---|
| Keyword rule | 0.148 | 0.606 | 0.238 |
| TF-IDF + LR | 0.000 | 0.000 | 0.000 |
| Qwen3-4B zero-shot | 0.344 | 0.939 | 0.504 |

**Conclusion:** supervised TF-IDF wins with plentiful labels encoding conventions; the zero-shot SLM wins when labels are scarce and the task needs reading comprehension. Next: QLoRA fine-tuning, to combine comprehension with learned conventions.

## Arm B: Qwen3-4B-Instruct-2507 fine-tuned with QLoRA (Unsloth, Colab T4)

Setup: 4-bit base, LoRA r=16 on attention and MLP projections, 1 epoch over 4,557 main-track training narratives (11 dropped: longer than 1,536 tokens), loss on the answer only, same prompt v1 as zero-shot. Training time: ~69 min.

| Model | Validation macro-F1 | Validation accuracy |
|---|---|---|
| Zero-shot | 0.594 | 0.698 |
| TF-IDF + LR | 0.716 | – |
| QLoRA fine-tuned | 0.801 | 0.875 |

- Fine-tuning fixed the diagnosed zero-shot errors: incorrect operation precision 0.48 → 0.86, excavation precision 0.34 → 0.87, equipment recall 0.61 → 0.91, natural force F1 0.52 → 0.76.
- Weakest: material failure and other outside force (F1 0.64, recall 0.56), small classes with blurry boundaries.
- One configuration, no validation-based tuning: validation is close to an unbiased estimate; test run once next.

### Final test results: main track (7 causes)

| Model | Macro-F1 | Accuracy | Inference |
|---|---|---|---|
| Dummy | 0.092 | 0.476 | – |
| Qwen3-4B zero-shot | 0.632 | 0.705 | GPU |
| TF-IDF + LR | 0.730 | 0.829 | CPU, near-instant |
| Qwen3-4B QLoRA fine-tuned | 0.841 | 0.881 | 0.58 s/narrative (T4) |

- Fine-tuning beats TF-IDF by +11 macro-F1 and +5 accuracy points. Test (0.841) ≥ validation (0.801): no sign of overfitting to older incidents.
- All classes ≥ 0.78 F1; material failure 0.79, other outside force 0.84.
- Cost: ~69 min one-time GPU training plus 0.58 s per prediction, trivial at PHMSA's volume (hundreds of incidents per year).

**Text A/B conclusion:** with plentiful labels, a fine-tuned SLM combines language understanding with learned conventions and beats TF-IDF; with scarce labels (pending detection), the zero-shot SLM wins. Zero-shot alone loses to supervised TF-IDF on the main task.

### Catastrophic forgetting: fine-tuned model on pending detection

| Model | Precision | Recall | F1 | Unclear answers |
|---|---|---|---|---|
| Zero-shot base | 0.344 | 0.939 | 0.504 | 0 |
| QLoRA fine-tuned | 0.000 | 0.000 | 0.000 | 59 |

- Trained only on narratives with a definite cause (unknowns excluded), the model learned that a cause always exists and never answers UNDETERMINED.
- 59 answers use neither word, likely cause letters: format drift toward the trained answer.
- Trade-off: +21 points on the trained task, −50 points on an untrained one.
- Design: LoRA adapters are separable, so route cause classification to base + adapter and pending detection to the base model alone (same GPU, no extra memory). Future: per-task adapters or multi-task fine-tuning.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import f1_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
R = ROOT / "reports"

tf = pd.read_parquet(R / "phmsa_main_tfidf_original_predictions.parquet")
ft = pd.read_parquet(R / "phmsa_main_finetuned_test.parquet")
j = tf.join(ft["pred"].rename("pred_ft"))
assert len(j) == len(tf) == len(ft), "test sets do not line up"

tf_ok, ft_ok = j["pred"] == j["label"], j["pred_ft"] == j["label"]
only_tf, only_ft = int((tf_ok & ~ft_ok).sum()), int((~tf_ok & ft_ok).sum())
p = binomtest(only_ft, only_tf + only_ft, 0.5).pvalue
print(f"Only TF-IDF right: {only_tf} | only fine-tuned right: {only_ft} | McNemar p = {p:.2g}")

rng = np.random.default_rng(42)
diffs = []
for _ in range(2000):
    s = j.iloc[rng.integers(0, len(j), len(j))]
    diffs.append(f1_score(s["label"], s["pred_ft"], average="macro") - f1_score(s["label"], s["pred"], average="macro"))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"Macro-F1 gain (fine-tuned − TF-IDF): 95% interval {lo:.3f} to {hi:.3f}")

Only TF-IDF right: 36 | only fine-tuned right: 75 | McNemar p = 0.00027
Macro-F1 gain (fine-tuned − TF-IDF): 95% interval 0.059 to 0.171
